# Train a microWakeWord model on Google Colab (Python 3.12)

This instructional notebook trains and exports a custom microWakeWord model. Start a **fresh Colab Python 3.12 GPU runtime**, then run cells from top to bottom. The default `SMOKE_TEST = True` exercises synthesis through TFLite export quickly; change it to `False` for useful training volumes. Model quality still requires substantially more data and tuning.

The dependency cell installs code from this fork's `fix/colab-python312` branch. Change `REPOSITORY_REF` only when intentionally testing another revision. No runtime restart is normally required because setup occurs before TensorFlow or PyTorch is imported. If Colab reports that a previously imported package cannot be replaced, restart once and resume at the diagnostics/verification cell.

In [ ]:
# User-configurable settings
SMOKE_TEST = True
TARGET_WORD = "khum_puter"  # phonetic spellings may synthesize better
REPOSITORY_URL = "https://github.com/CJMarais/micro-wake-word.git"
REPOSITORY_REF = "fix/colab-python312"

# Smoke mode proves the pipeline, not model quality.
SYNTHETIC_SAMPLE_COUNT = 24 if SMOKE_TEST else 1000
TRAINING_STEPS = 2 if SMOKE_TEST else 10000
BATCH_SIZE = 8 if SMOKE_TEST else 128
MAX_AUGMENTATION_FILES = 8 if SMOKE_TEST else None

In [ ]:
# Environment diagnostics (keep this output with bug reports)
import os, platform, shutil, subprocess, sys
from pathlib import Path

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Executable:", sys.executable)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("This Colab notebook requires Python 3.12.")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], check=True)
else:
    print("nvidia-smi: unavailable (select a GPU runtime for normal training)")

In [ ]:
# Environment/dependency implementation details
from pathlib import Path
import os, shutil, subprocess, sys

WORKSPACE = Path("/content/micro-wake-word-work") if Path("/content").is_dir() else Path.cwd() / ".notebook-work"
REPO_DIR = WORKSPACE / "micro-wake-word"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, *, cwd=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run([str(part) for part in command], cwd=cwd, check=True, text=True)

# Explicit codecs are required for FLAC/MP3 conversion in fresh Colab images.
if Path("/content").is_dir():
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1"])

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, REPO_DIR])
else:
    print("Reusing existing checkout:", REPO_DIR)

requirements = REPO_DIR / "notebooks" / "requirements_colab_py312.txt"
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", requirements])
# Dependencies were resolved above as one coherent set; install this exact checkout.
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", REPO_DIR])
os.chdir(WORKSPACE)
print("Working directory:", Path.cwd())

In [ ]:
# Verify the resolved runtime before doing expensive work.
import datasets, numpy as np, tensorflow as tf, torch, torchaudio
import audiomentations, scipy
from importlib.metadata import version

print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__, "GPUs:", tf.config.list_physical_devices("GPU"))
print("PyTorch:", torch.__version__, "torchaudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available(), "CUDA:", torch.version.cuda)
print("datasets:", datasets.__version__, "audiomentations:", version("audiomentations"))
print("Piper Sample Generator:", version("piper-sample-generator"))
assert np.lib.NumpyVersion(np.__version__) >= np.lib.NumpyVersion("2.0.0")
assert tf.__version__.startswith("2.21.")

## Synthesize and verify the wake word

The first command generates **exactly one** WAV and validates it before any bulk generation. Piper 3.2.0's installed module entry point replaces the historical cloned `generate_samples.py` script.

In [ ]:
from IPython.display import Audio, display
from pathlib import Path
import shutil, subprocess, sys, wave

PIPER_MODEL_DIR = WORKSPACE / "piper-models"
PIPER_MODEL = PIPER_MODEL_DIR / "en_US-libritts_r-medium.pt"
PIPER_CONFIG = Path(str(PIPER_MODEL) + ".json")
SAMPLE_DIR = WORKSPACE / "generated_samples"
PIPER_MODEL_DIR.mkdir(exist_ok=True)
if not PIPER_MODEL.exists():
    run(["wget", "-q", "--show-progress", "-O", PIPER_MODEL, "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"])
if not PIPER_CONFIG.exists():
    run(["wget", "-q", "--show-progress", "-O", PIPER_CONFIG, "https://raw.githubusercontent.com/rhasspy/piper-sample-generator/v3.2.0/models/en_US-libritts_r-medium.pt.json"])
assert PIPER_MODEL.stat().st_size > 0 and PIPER_CONFIG.stat().st_size > 0, "Piper model/config download is empty"

smoke_dir = WORKSPACE / "piper_one_sample"
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
run([sys.executable, "-m", "piper_sample_generator", TARGET_WORD, "--model", PIPER_MODEL, "--max-samples", "1", "--batch-size", "1", "--output-dir", smoke_dir])
smoke_wavs = list(smoke_dir.glob("*.wav"))
assert len(smoke_wavs) == 1 and smoke_wavs[0].stat().st_size > 44, f"Expected exactly one valid WAV, found {smoke_wavs}"
with wave.open(str(smoke_wavs[0]), "rb") as wav:
    assert wav.getnframes() > 0 and wav.getframerate() > 0
display(Audio(filename=str(smoke_wavs[0]), autoplay=True))

In [ ]:
# Bulk sample generation (idempotent: regenerate only if the count is short).
SAMPLE_DIR.mkdir(exist_ok=True)
existing = list(SAMPLE_DIR.glob("*.wav"))
if len(existing) < SYNTHETIC_SAMPLE_COUNT:
    shutil.rmtree(SAMPLE_DIR)
    batch = min(100, SYNTHETIC_SAMPLE_COUNT)
    run([sys.executable, "-m", "piper_sample_generator", TARGET_WORD, "--model", PIPER_MODEL, "--max-samples", SYNTHETIC_SAMPLE_COUNT, "--batch-size", batch, "--output-dir", SAMPLE_DIR])
generated_wavs = sorted(SAMPLE_DIR.glob("*.wav"))
assert len(generated_wavs) == SYNTHETIC_SAMPLE_COUNT
assert all(path.stat().st_size > 44 for path in generated_wavs)
print(f"Validated {len(generated_wavs)} synthesized WAV files")

## Obtain augmentation audio

MIT room impulse responses are always downloaded. Full mode also downloads the historical AudioSet FLAC and FMA MP3 subsets and converts them explicitly with FFmpeg to mono 16 kHz PCM WAV. Smoke mode creates small deterministic noise backgrounds so the entire pipeline can be validated without multi-gigabyte downloads; full mode is the dataset/codec integration path for real training. Review the source dataset licences before distributing or commercially using a model.

In [ ]:
import datasets, numpy as np, scipy.io.wavfile, soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm
import shutil, tarfile, urllib.request, zipfile

RIR_DIR = WORKSPACE / "mit_rirs"
BACKGROUND_DIR = WORKSPACE / "background_16k"
RIR_DIR.mkdir(exist_ok=True)
BACKGROUND_DIR.mkdir(exist_ok=True)

if not list(RIR_DIR.glob("*.wav")):
    rir_rows = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for index, row in enumerate(tqdm(rir_rows, desc="MIT RIR")):
        audio = row["audio"]
        samples = np.asarray(audio["array"], dtype=np.float32)
        assert samples.size and np.isfinite(samples).all()
        name = Path(audio["path"]).stem + ".wav"
        scipy.io.wavfile.write(RIR_DIR / name, 16000, np.clip(samples * 32767, -32768, 32767).astype(np.int16))
        if MAX_AUGMENTATION_FILES and index + 1 >= MAX_AUGMENTATION_FILES:
            break

def download(url, destination):
    destination = Path(destination)
    if not destination.exists() or destination.stat().st_size == 0:
        urllib.request.urlretrieve(url, destination)
    assert destination.stat().st_size > 0

def convert_audio(inputs, output_dir, limit=None):
    converted = []
    for source in list(inputs)[:limit]:
        destination = output_dir / (source.stem + ".wav")
        if not destination.exists():
            run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", source, "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", "-y", destination])
        samples, rate = sf.read(destination, dtype="float32")
        assert rate == 16000 and samples.size and np.isfinite(samples).all()
        converted.append(destination)
    return converted

if SMOKE_TEST:
    rng = np.random.default_rng(312)
    for index in range(8):
        path = BACKGROUND_DIR / f"smoke_noise_{index}.wav"
        if not path.exists():
            scipy.io.wavfile.write(path, 16000, (rng.normal(0, 0.04, 48000) * 32767).astype(np.int16))
else:
    downloads = WORKSPACE / "downloads"
    downloads.mkdir(exist_ok=True)
    audioset_tar = downloads / "bal_train09.tar"
    download("https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar", audioset_tar)
    audioset_raw = WORKSPACE / "audioset_raw"
    if not audioset_raw.exists():
        audioset_raw.mkdir(); tarfile.open(audioset_tar).extractall(audioset_raw, filter="data")
    assert convert_audio(audioset_raw.rglob("*.flac"), BACKGROUND_DIR)
    fma_zip = downloads / "fma_xs.zip"
    download("https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip", fma_zip)
    fma_raw = WORKSPACE / "fma_raw"
    if not fma_raw.exists():
        fma_raw.mkdir(); zipfile.ZipFile(fma_zip).extractall(fma_raw)
    assert convert_audio(fma_raw.rglob("*.mp3"), BACKGROUND_DIR)

assert list(RIR_DIR.glob("*.wav")) and list(BACKGROUND_DIR.glob("*.wav"))
print("RIR files:", len(list(RIR_DIR.glob("*.wav"))), "background files:", len(list(BACKGROUND_DIR.glob("*.wav"))))

In [ ]:
# Configure and smoke-test the actual microWakeWord audio pipeline.
from IPython.display import Audio, display
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.audio_utils import save_clip
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(str(SAMPLE_DIR), "*.wav", remove_silence=False, random_split_seed=10, split_count=0.1)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={"SevenBandParametricEQ": .1, "TanhDistortion": .1, "PitchShift": .1, "BandStopFilter": .1, "AddColorNoise": .1, "AddBackgroundNoise": .75, "Gain": 1.0, "RIR": .5},
    impulse_paths=[str(RIR_DIR)], background_paths=[str(BACKGROUND_DIR)],
    background_min_snr_db=-5, background_max_snr_db=10, min_jitter_s=.195, max_jitter_s=.205,
)
augmented_clip = augmenter.augment_clip(clips.get_random_clip())
assert augmented_clip.shape == (51200,) and np.isfinite(augmented_clip).all()
AUGMENTED_WAV = WORKSPACE / "augmented_clip.wav"
save_clip(augmented_clip, str(AUGMENTED_WAV))
display(Audio(filename=str(AUGMENTED_WAV), autoplay=True))

feature_probe = SpectrogramGeneration(clips, augmenter, slide_frames=1, step_ms=10).get_random_spectrogram()
assert feature_probe.ndim == 2 and feature_probe.shape[0] > 0 and np.isfinite(feature_probe).all()
print("Feature smoke test shape:", feature_probe.shape)

In [ ]:
# Generate positive train/validation/test RaggedMmap features.
from mmap_ninja.ragged import RaggedMmap
import shutil

POSITIVE_FEATURES = WORKSPACE / "generated_augmented_features"
if POSITIVE_FEATURES.exists():
    shutil.rmtree(POSITIVE_FEATURES)
for split, source_split, repetition, slide_frames in [( "training", "train", 1 if SMOKE_TEST else 2, 2 if SMOKE_TEST else 10), ("validation", "validation", 1, 1), ("testing", "test", 1, 1)]:
    out_dir = POSITIVE_FEATURES / split / "wakeword_mmap"
    spectrograms = SpectrogramGeneration(clips, augmenter, slide_frames=slide_frames, step_ms=10)
    RaggedMmap.from_generator(out_dir=out_dir, sample_generator=spectrograms.spectrogram_generator(split=source_split, repeat=repetition), batch_size=8 if SMOKE_TEST else 100, verbose=True)
    assert out_dir.exists() and any(out_dir.iterdir())
print("Positive feature sets created at", POSITIVE_FEATURES)

In [ ]:
# Obtain negative features. Smoke mode builds a tiny local set; full mode uses
# the pre-generated microWakeWord feature archives.
import shutil, urllib.request, zipfile

NEGATIVE_FEATURES = WORKSPACE / "negative_datasets"
if SMOKE_TEST:
    if NEGATIVE_FEATURES.exists(): shutil.rmtree(NEGATIVE_FEATURES)
    background_clips = Clips(str(BACKGROUND_DIR), "*.wav", random_split_seed=11, split_count=0.1)
    for split, source_split in [("training", "train"), ("validation", "validation"), ("testing", "test")]:
        out_dir = NEGATIVE_FEATURES / "smoke_noise" / split / "noise_mmap"
        generator = SpectrogramGeneration(background_clips, split_spectrogram_duration_s=3.2, step_ms=10).spectrogram_generator(split=source_split)
        RaggedMmap.from_generator(out_dir=out_dir, sample_generator=generator, batch_size=8, verbose=True)
else:
    NEGATIVE_FEATURES.mkdir(exist_ok=True)
    root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    for name in ["dinner_party.zip", "dinner_party_eval.zip", "no_speech.zip", "speech.zip"]:
        archive = NEGATIVE_FEATURES / name
        download(root + name, archive)
        marker = NEGATIVE_FEATURES / (name + ".extracted")
        if not marker.exists():
            zipfile.ZipFile(archive).extractall(NEGATIVE_FEATURES); marker.touch()
assert any(NEGATIVE_FEATURES.rglob("*mmap")), "No negative feature mmap was created/downloaded"

In [ ]:
# Write the training configuration. User tuning belongs here, not in setup.
import yaml
TRAIN_DIR = WORKSPACE / "trained_models" / "wakeword"
positive_feature = {"features_dir": str(POSITIVE_FEATURES), "sampling_weight": 2.0, "penalty_weight": 1.0, "truth": True, "truncation_strategy": "truncate_start", "type": "mmap"}
if SMOKE_TEST:
    negative_features = [{"features_dir": str(NEGATIVE_FEATURES / "smoke_noise"), "sampling_weight": 1.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"}]
else:
    negative_features = [
      {"features_dir": str(NEGATIVE_FEATURES / "speech"), "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
      {"features_dir": str(NEGATIVE_FEATURES / "dinner_party"), "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
      {"features_dir": str(NEGATIVE_FEATURES / "no_speech"), "sampling_weight": 5.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
      {"features_dir": str(NEGATIVE_FEATURES / "dinner_party_eval"), "sampling_weight": 0.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "split", "type": "mmap"},
    ]
config = {
 "window_step_ms": 10, "train_dir": str(TRAIN_DIR), "features": [positive_feature, *negative_features],
 "training_steps": [TRAINING_STEPS], "positive_class_weight": [1], "negative_class_weight": [20],
 "learning_rates": [0.001], "batch_size": BATCH_SIZE, "time_mask_max_size": [0], "time_mask_count": [0],
 "freq_mask_max_size": [0], "freq_mask_count": [0], "eval_step_interval": 1 if SMOKE_TEST else 500,
 "clip_duration_ms": 1500, "target_minimization": 0.9, "minimization_metric": None, "maximization_metric": "average_viable_recall",
}
TRAINING_CONFIG = WORKSPACE / "training_parameters.yaml"
TRAINING_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(TRAINING_CONFIG.read_text())

In [ ]:
# Train, evaluate, convert, and fail at the root command if anything breaks.
weights_to_export = "last_weights" if SMOKE_TEST else "best_weights"
training_command = [sys.executable, "-m", "microwakeword.model_train_eval",
 "--training_config", TRAINING_CONFIG, "--train", "1", "--restore_checkpoint", "0" if SMOKE_TEST else "1",
 "--test_tf_nonstreaming", "0", "--test_tflite_nonstreaming", "0", "--test_tflite_nonstreaming_quantized", "0",
 "--test_tflite_streaming", "0", "--test_tflite_streaming_quantized", "1", "--use_weights", weights_to_export,
 "mixednet", "--pointwise_filters", "64,64,64,64", "--repeat_in_block", "1, 1, 1, 1",
 "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]", "--residual_connection", "0,0,0,0",
 "--first_conv_filters", "32", "--first_conv_kernel_size", "5", "--stride", "3"]
run(training_command)
TFLITE_MODEL = TRAIN_DIR / "tflite_stream_state_internal_quant" / "stream_state_internal_quant.tflite"
assert TFLITE_MODEL.exists() and TFLITE_MODEL.stat().st_size > 0, f"Missing/empty export: {TFLITE_MODEL}"
print("Valid TFLite export:", TFLITE_MODEL, TFLITE_MODEL.stat().st_size, "bytes")

In [ ]:
# Download the verified model when running in Colab.
try:
    from google.colab import files
except ImportError:
    print("Model is available at", TFLITE_MODEL)
else:
    files.download(str(TFLITE_MODEL))

# ESPHome also needs a model manifest. See:
# https://esphome.io/components/micro_wake_word/
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2